# Lab 3.1 — Resume Chatbot with Gradio

## Introduction

This is the **first lab** in the Resume Chatbot module. You will build a simple chatbot that represents you on a personal website — answering questions about your career, skills, and experience based on your resume.

The lab combines several ideas from earlier modules:
- **LLM calls** with a system prompt (Foundations)
- **Document loading** — extract text from a PDF resume using `pypdf`
- **A chat UI** — wrap your logic in a Gradio `ChatInterface` so anyone can talk to your bot in the browser

By the end, you will have a working chatbot that stays in character as you and answers questions grounded in your resume content.

Before starting, make sure you have:
- Completed the Foundations and Setup labs
- `OPENAI_API_KEY` and `YOUR_NAME` set in your `.env` file
- Your resume placed at `me/my-resume.pdf` (or update the path in the code)

---

## Intention (Learning Objectives)

By the end of this lab, you should be able to:

1. **Extract text from a PDF** — use `pypdf` to read resume content into a Python string
2. **Build a persona system prompt** — inject your name and resume into a prompt that defines the chatbot's character
3. **Maintain conversation history** — pass Gradio's `history` into the OpenAI `messages` list on each turn
4. **Create a chat function** — wire up a `chat(message, history)` handler that calls `gpt-4o-mini`
5. **Launch a Gradio UI** — expose your chatbot through `gr.ChatInterface` in the browser

---

## Lab Structure

| Part | Content |
|------|---------|
| **Part 1** | Imports, load environment variables, create OpenAI client |
| **Part 2** | Extract resume text from PDF |
| **Part 3** | Load your name from environment variables |
| **Part 4** | Build the system prompt with resume context |
| **Part 5** | Implement the chat function and launch Gradio |

> **Note:** Run cells top to bottom in order. Replace `me/my-resume.pdf` with your own resume before demoing.

## Part 1 — Setup

Import dependencies, load your `.env` file, and create the OpenAI client.

In [8]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr
import os

In [2]:
load_dotenv(override=True)
openai = OpenAI()

## Part 2 — Load Resume from PDF

Use `pypdf` to extract text from every page of your resume PDF. This text becomes the knowledge base for your chatbot.

In [3]:
reader = PdfReader("me/my-resume.pdf")
resume = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        resume += text

In [4]:
print(resume)

John Rippin | Node.js Developer
Creative Node.js Developer with 12+ years of strong experience in Operating Systems & Terminal, 
Machine Learning areas with willingness to learn and master Scrum & Agile and RDBMS & SQL. 
Blockchain expert.
>> Download Word, DOCX Version Now <<
  
>> Download Word, DOCX Version Now <<
CONTACTS
• Phone: (997) 360-2706
• Email: john11@gmail.com
• LinkedIn: @john11
• Github: @john11
KEY SKILLS
Front End:
• BabylonJS, Backbone.js, Handlebars.js
• JavaScript/ES6/ES2017, MeteorJS, RESTful API
• ReactJS, Underscore.js, Web Components
• WebGL, XHTML, XML
Back End:
• Node.js 0.8-10.1, Azure Functions, Flask
• LINQ, MS Build, kops
Databases:
• CRUD, Cassandra, DAX
• MongoDB, MySQL, NHibernate
• SQLLite, SSIS, SSRS
Testing:
• Blanket, Nock, UnexpectedDevOps:
• CI/CD, Jenkins, Raygun, Travis CI
SDLC:
• 12 Factor App, Asana, Basecamp
• Bitbucket, Crucible, ER Diagrams
• Lint, SOLID principles, UML
WORK HISTORY
Mayer LLC, Senior Node.js Developer, 01/2014 - 12/2016 |

## Part 3 — Load Your Name

Read `YOUR_NAME` from the environment. The chatbot will use this to stay in character.

In [10]:
# For your data
your_name = os.getenv("YOUR_NAME")

if your_name:
    print(f"Your Email found and starts with {your_name[0:5]}")
else:
    print("Your Email not found")


name = your_name

Your Email found and starts with Aphir


## Part 4 — Build the System Prompt

Combine your name and resume into a system prompt that tells the LLM who it is representing and what context it can use to answer questions.

In [11]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Resume:\n{resume}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [12]:
system_prompt

'You are acting as Aphirat Nimanussonkul. You are answering questions on Aphirat Nimanussonkul\'s website, particularly questions related to Aphirat Nimanussonkul\'s career, background, skills and experience. Your responsibility is to represent Aphirat Nimanussonkul for interactions on the website as faithfully as possible. You are given a summary of Aphirat Nimanussonkul\'s background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don\'t know the answer, say so.\n\n## Resume:\n## John Rippin | Node.js Developer\n\nCreative Node.js Developer with 12+ years of strong experience in Operating Systems & Terminal, Machine Learning areas with willingness to learn and master Scrum & Agile and RDBMS & SQL. Blockchain expert.\n\n[>> Download Word, DOCX Version Now <<](https://www.fullstackresume.com/blog/node-js-developer-resume-sample)\n<table>\n <tr>\n  <td>\n 

## Part 5 — Chat Function & Gradio UI

Implement a `chat` function that prepends the system prompt and conversation history to each user message, then launch the Gradio chat interface.

In [13]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [14]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


---

## Next Steps

Try asking your chatbot questions about your experience, skills, and background. In [Lab 3.2](3_lab2.ipynb), you will extend this chatbot with **tool calling** and email notifications so it can record leads and unknown questions.